In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
data_path = Path("../data/raw/water_interruptions_raw.csv")
if not data_path.exists():
    data_path = Path("data/raw/water_interruptions_raw.csv")

df = pd.read_csv(data_path)

In [ ]:
df = df.rename(columns={
    "start_datetime": "interruption_start_datetime",
    "restoration_datetime": "expected_restoration_datetime",
    "duration_hours": "expected_duration_hours"
})

In [ ]:
df["interruption_start_datetime"] = pd.to_datetime(
    df["interruption_start_datetime"]
)

df["expected_restoration_datetime"] = pd.to_datetime(
    df["expected_restoration_datetime"]
)

In [ ]:
df["calculated_expected_duration_hours"] = (
    df["expected_restoration_datetime"]
    - df["interruption_start_datetime"]
).dt.total_seconds() / 3600

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df["event_id"].nunique()

In [ ]:
df["event_id"].duplicated().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
pd.Series(np.isclose(
    df["expected_duration_hours"],
    df["calculated_expected_duration_hours"]
)).value_counts()

In [ ]:
df[
    [
        "expected_duration_hours",
        "calculated_expected_duration_hours"
    ]
]

In [ ]:
df[
    [
        "subscribers_affected",
        "calculated_expected_duration_hours"
    ]
].describe()

In [ ]:
df["service_center"].value_counts()

In [ ]:
df["work_type"].value_counts()

In [ ]:
df.nlargest(
    5,
    "subscribers_affected"
)[
    [
        "service_center",
        "work_type",
        "subscribers_affected",
        "calculated_expected_duration_hours"
    ]
]

In [ ]:
df.nlargest(
    5,
    "calculated_expected_duration_hours"
)[
    [
        "service_center",
        "work_type",
        "calculated_expected_duration_hours",
        "subscribers_affected"
    ]
]

In [ ]:
df["expected_subscriber_hours"] = (
    df["subscribers_affected"]
    * df["calculated_expected_duration_hours"]
)

In [ ]:
df.nlargest(
    5,
    "expected_subscriber_hours"
)[
    [
        "service_center",
        "work_type",
        "subscribers_affected",
        "calculated_expected_duration_hours",
        "expected_subscriber_hours"
    ]
]

## Initial Findings

This notebook contains an initial inspection of the pilot
water interruption dataset.

### Data quality

- 12 records were analyzed.
- Event IDs are unique.
- No null values were found in the pilot dataset.
- Start and expected restoration timestamps were converted to datetime.
- Reported expected durations were independently recalculated and validated.

### Preliminary observation

The largest interruption by affected subscribers was not
the largest interruption when expected duration was also considered.

To capture both dimensions, I created:

**Expected Subscriber-Hours = Affected Subscribers × Expected Duration**

In the pilot dataset, one Telavi interruption affected 3,441
subscribers for an expected 2 hours (6,882 expected subscriber-hours),
while one Tsalenjikha interruption affected 2,052 subscribers for an
expected 24 hours (49,248 expected subscriber-hours).

### Important limitation

Expected restoration time does not necessarily represent actual
restoration time. Therefore, this metric measures expected rather
than confirmed service disruption.

The pilot dataset is too small to make municipality-level conclusions.
The next step is to collect a substantially larger historical dataset.